### Split count table into three sets
======================================================== <br>

- One summing the counts for all read lengths per sample. This one will be used for DT analysis when focusing on genes
- The two sets summing 20-23nt and keeping 24nt separately. This two will be used for DT analysis when focusing on transposable elements (TEs). The reasoning behind keeping these length classes seperatly is that they regulate TEs differently.

First load the required libraries, set the working directory and load a R script with custom functions used.


In [1]:
import numpy as np
import polars as pl

Load the raw table of counts

The rows in the table represent small RNA peaks across the reference genome used. 

Each sample has seperat counts for the small RNA read lenghts 20-24 nt

In [2]:
# load raw count data
df = (pl.read_csv("/home/mier0006/Documents/phd_dact/smRNA_dact/01_counts/dactylorhiza_sRNA-read-counts_MM-fraqtion.txt", separator="\t"))
# replace the . in the sample names with - to avoid problems later on
df = (df
      .rename({x:x.replace(".", "_") for x in df.columns})
)
df.head()

peakID,fB1_1804_20nt,fB1_1804_21nt,fB1_1804_22nt,fB1_1804_23nt,fB1_1804_24nt,fB5_1855_20nt,fB5_1855_21nt,fB5_1855_22nt,fB5_1855_23nt,fB5_1855_24nt,fP1_1001_20nt,fP1_1001_21nt,fP1_1001_22nt,fP1_1001_23nt,fP1_1001_24nt,fPx_1707n_20nt,fPx_1707n_21nt,fPx_1707n_22nt,fPx_1707n_23nt,fPx_1707n_24nt,iA11_1586_20nt,iA11_1586_21nt,iA11_1586_22nt,iA11_1586_23nt,iA11_1586_24nt,iB5_1870_20nt,iB5_1870_21nt,iB5_1870_22nt,iB5_1870_23nt,iB5_1870_24nt,iS3_1904_20nt,iS3_1904_21nt,iS3_1904_22nt,iS3_1904_23nt,iS3_1904_24nt,iS4_1908_20nt,…,tB2_1805_23nt,tB2_1805_24nt,tB3_1812_20nt,tB3_1812_21nt,tB3_1812_22nt,tB3_1812_23nt,tB3_1812_24nt,tB5_1826_20nt,tB5_1826_21nt,tB5_1826_22nt,tB5_1826_23nt,tB5_1826_24nt,tB5_1830_20nt,tB5_1830_21nt,tB5_1830_22nt,tB5_1830_23nt,tB5_1830_24nt,tB5_1833_20nt,tB5_1833_21nt,tB5_1833_22nt,tB5_1833_23nt,tB5_1833_24nt,tS3_1901n_20nt,tS3_1901n_21nt,tS3_1901n_22nt,tS3_1901n_23nt,tS3_1901n_24nt,tS3_1902_20nt,tS3_1902_21nt,tS3_1902_22nt,tS3_1902_23nt,tS3_1902_24nt,tS5_1918_20nt,tS5_1918_21nt,tS5_1918_22nt,tS5_1918_23nt,tS5_1918_24nt
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""scaffold1_42451085:10002001-10…",0.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1,0.0,0.0,0.0,0.0,0.2,0.0,0.5,0.14,0.0,0.0,0.0,0.0,0.0,0.0,0.12,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,…,1.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.0,0.0,0.0,1.0,0.25,0.5,0.0,0.1,0.0,0.2,0.0
"""scaffold1_42451085:10003301-10…",0.0,0.14,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.07,0.0,0.0,0.0,0.0,0.05,0.0,0.17,0.0,0.0,0.0,0.0,0.5,0.07,0.14,0.14,0.0,0.05,0.0,0.0,0.07,0.0,0.0,0.0,0.14,0.43,0.0,0.0
"""scaffold1_42451085:10003501-10…",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.14,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.39,0.0,0.0,0.0,0.07,0.39,0.0,…,0.0,0.0,0.0,0.0,0.17,0.1,0.5,0.2,0.0,0.14,0.0,0.22,0.14,0.06,0.14,0.0,0.25,0.0,0.0,0.25,0.0,0.95,0.0,0.0,0.0,0.0,0.5,0.0,0.0,0.0,0.0,0.07,0.0,0.0,0.0,0.0,0.05
"""scaffold1_42451085:1000401-100…",1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.33,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.04,0.0,0.0,0.0,1.0,…,0.0,0.0,0.33,0.33,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.67,0.33,0.0,0.0,0.0,0.33,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.67,0.0,0.0,0.0,0.0,0.33,0.04,0.0,0.0,0.0
"""scaffold1_42451085:10004401-10…",0.0,0.0,0.0,0.07,0.04,0.0,0.03,0.0,0.0,0.09,0.17,0.07,0.0,0.0,0.03,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.05,0.1,0.0,0.0,0.03,0.0,0.2,0.0,0.52,0.0,0.02,0.14,0.0,…,0.0,0.0,0.04,0.17,0.03,0.15,0.22,0.1,0.03,0.12,0.07,0.04,0.12,0.02,0.04,0.1,0.41,0.0,0.07,0.07,0.03,0.05,0.0,0.0,0.0,0.0,0.12,0.0,0.0,0.05,0.02,0.05,0.05,0.03,0.06,0.0,0.0


Make an array with the unique sample ids

In [7]:
uniq_samples = np.unique(np.array(["_".join(x.split("_")[0:2]) for x in df.columns if x != "peakID"]))
uniq_samples

array(['fB1_1804', 'fB5_1855', 'fP1_1001', 'fPx_1707n', 'iA11_1586',
       'iB5_1870', 'iS3_1904', 'iS4_1908', 'mA10_1573c', 'mA13_1661',
       'mA15_1775', 'mA1_1567', 'mP1_1722n', 'mP3_1748', 'mS1_1765',
       'mS2_1757', 'mS2_1760', 'tA13_1670', 'tA9_1553c', 'tA9_1641',
       'tB2_1798', 'tB2_1805', 'tB3_1812', 'tB5_1826', 'tB5_1830',
       'tB5_1833', 'tS3_1901n', 'tS3_1902', 'tS5_1918'], dtype='<U10')

**Sum the 20-24nt counts for each sample** 
- For each unique sample id select the columns starting with it and sum the counts into a new column named by the sample id.
- Select only the `peakID` and uniq sample id columns
- Write the table to a file.


In [8]:
gene_20_24_filepath = "/home/mier0006/Documents/phd_dact/smRNA_dact/01_counts/dactylorhiza_sRNA-read-counts_MM-fraction_per-sample-summed-20-24nt.txt"
(df
 .with_columns((pl.concat_list([x for x in df.columns if x.startswith(y)]).list.sum().alias(y) for y in uniq_samples))
 .select(["peakID"] + uniq_samples.tolist())
 ).write_csv(gene_20_24_filepath, separator="\t")

**Sum the 20-23nt counts for each sample** 
- Remove the 24nt counts from the table. 
- For each unique sample id select the columns starting with it and sum the counts into a new column named by the sample id. 
- Select only the `peakID` and uniq sample id columns 
- Write the table to a file.

In [9]:
te_20_23_filepath = "/home/mier0006/Documents/phd_dact/smRNA_dact/01_counts/dactylorhiza_sRNA-read-counts_MM-fraction_per-sample-summed-20-23nt.txt"
(df
 .with_columns((pl.concat_list([i for i in df.columns if i.startswith(y) and not i.endswith("24nt")]).list.sum().alias(y) for y in uniq_samples))
 .select(["peakID"] + uniq_samples.tolist())
 ).write_csv(te_20_23_filepath, separator="\t")

**Extract 24nt counts for each sample** 
- Select columns ending in 24nt
- Rename removing `_24nt`from the column names
- Write the table to a file.

In [3]:
te_24_filepath = "/home/mier0006/Documents/phd_dact/smRNA_dact/01_counts/dactylorhiza_sRNA-read-counts_MM-fraction_per-sample-24nt.txt"
(df
 .select(pl.col("peakID"), pl.col("^.*24nt$"))
 .rename({x:x.replace("_24nt", "") for x in df.columns if x != "peakID"})
 ).write_csv(te_24_filepath, separator="\t")